# Topic 18: Naive Bayes: Probability by Counting, Bayes' Rule, and Gaussian Naive Bayes

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 18 lecture note](https://dongchul.kim/csci3320-fall2026/topic/18b), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## Training is counting

In [ ]:
import pandas as pd

rows = [
    ("Sunny", "Hot", "High", "Weak", "No"),        ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),     ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),  ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),  ("Rain", "Mild", "High", "Strong", "No"),
]
df = pd.DataFrame(rows, columns=["Outlook", "Temp", "Humidity", "Wind", "Play"])

print(df["Play"].value_counts(), "\n")           # the priors: 9 Yes, 5 No
for f in ["Outlook", "Temp", "Humidity", "Wind"]:
    print(pd.crosstab(df[f], df["Play"]), "\n")  # one count table per feature

## Day 15, worked by hand

In [ ]:
import pandas as pd

rows = [
    ("Sunny", "Hot", "High", "Weak", "No"),        ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),     ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),  ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),  ("Rain", "Mild", "High", "Strong", "No"),
]
df = pd.DataFrame(rows, columns=["Outlook", "Temp", "Humidity", "Wind", "Play"])
day15 = {"Outlook": "Sunny", "Temp": "Cool", "Humidity": "High", "Wind": "Strong"}

scores = {}
for cls in ["Yes", "No"]:
    days = df[df["Play"] == cls]                 # only the days of this class
    score = len(days) / len(df)                  # prior
    for f, value in day15.items():
        score *= (days[f] == value).sum() / len(days)   # likelihood of one feature
    scores[cls] = score
    print(f"{cls:3s} score {score:.5f}")

total = sum(scores.values())
for cls, s in scores.items():
    print(f"P({cls} | day 15) = {s / total:.3f}")

## The zero problem, and the +1 fix

In [ ]:
import pandas as pd

rows = [
    ("Sunny", "Hot", "High", "Weak", "No"),        ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),     ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),  ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),  ("Rain", "Mild", "High", "Strong", "No"),
]
df = pd.DataFrame(rows, columns=["Outlook", "Temp", "Humidity", "Wind", "Play"])
day16 = {"Outlook": "Overcast", "Temp": "Hot", "Humidity": "High", "Wind": "Weak"}

def posterior(day, alpha):
    scores = {}
    for cls in ["Yes", "No"]:
        days = df[df["Play"] == cls]
        score = len(days) / len(df)
        for f, value in day.items():
            k = df[f].nunique()                  # how many values this feature has
            score *= ((days[f] == value).sum() + alpha) / (len(days) + alpha * k)
        scores[cls] = score
    total = sum(scores.values())
    return {cls: round(float(s / total), 3) for cls, s in scores.items()}

print("no smoothing:", posterior(day16, alpha=0))
print("+1 smoothing:", posterior(day16, alpha=1))

## Naive Bayes in scikit-learn

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

rows = [
    ("Sunny", "Hot", "High", "Weak", "No"),        ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),     ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"), ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),  ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),  ("Rain", "Mild", "High", "Strong", "No"),
]
feats = ["Outlook", "Temp", "Humidity", "Wind"]
df = pd.DataFrame(rows, columns=feats + ["Play"])

enc = OrdinalEncoder()                       # categories -> codes 0, 1, 2
X = enc.fit_transform(df[feats])
y = df["Play"]

nb = CategoricalNB(alpha=1e-10)              # almost no smoothing, to match the hand count
nb.fit(X, y)                                 # = count the tables

day15 = pd.DataFrame([["Sunny", "Cool", "High", "Strong"]], columns=feats)
print(nb.classes_)
print(nb.predict_proba(enc.transform(day15)))

nb1 = CategoricalNB(alpha=1.0).fit(X, y)     # the default: +1 smoothing
day16 = pd.DataFrame([["Overcast", "Hot", "High", "Weak"]], columns=feats)
print(nb1.predict_proba(enc.transform(day16)))

## Numbers: Gaussian Naive Bayes

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

gnb = GaussianNB()
gnb.fit(X_tr, y_tr)                          # a mean and a variance per class and feature
print("training accuracy:", round(gnb.score(X_tr, y_tr), 3))
print("test accuracy:    ", round(gnb.score(X_te, y_te), 3))

i = list(data.feature_names).index("worst radius")
print("worst radius, mean per class:", gnb.theta_[:, i].round(2))       # row 0 malignant, 1 benign
print("worst radius, sd per class:  ", (gnb.var_[:, i] ** 0.5).round(2))
p = gnb.predict_proba(X_te[:1])[0]              # first test patient (truly malignant)
print(f"first test patient: P(malignant) = {p[0]:.5f}, P(benign) = {p[1]:.5f}")

## Two things to know about Naive Bayes

### 2. Multiplying many small numbers

In [ ]:
import math

p = 0.01
print("30 factors: ", p ** 30)
print("200 factors:", p ** 200)                       # below the smallest float: 0.0
print("sum of 200 logs:", 200 * math.log(p))          # no problem at all

## Side by side on Breast cancer

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)
models = {
    "tree, no limit":      lambda: DecisionTreeClassifier(random_state=0),
    "tree, max_depth=4":   lambda: DecisionTreeClassifier(max_depth=4, random_state=0),
    "Gaussian NB":         lambda: GaussianNB(),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}
acc = {name: [] for name in models}
for seed in range(5):                        # five different 80/20 splits
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed)
    for name, make in models.items():
        model = make().fit(X_tr, y_tr)
        acc[name].append(model.score(X_te, y_te))

for name, a in acc.items():
    print(f"{name:20s} {np.mean(a):.3f} +- {np.std(a):.3f}")

Back to the [Topic 18 lecture note](https://dongchul.kim/csci3320-fall2026/topic/18b).